# Invariant Coordinate Selection (ICS) - Julia tutorial
## ICS and Related Methods Conference, 26-28 May 2026, Helsinki, Finland
Valentin Todorov, valentin@todorov.at

## Introduction
Invariant Coordinate Selection (ICS) is an unsupervised multivariate statistical method used for dimension reduction, outlier detection, and cluster identification. By simultaneously diagonalizing two different scatter matrices, ICS projects high-dimensional data into a new, meaningful coordinate system to easily uncover underlying structures.
It is particularly useful for dimension reduction. Unlike PCA, ICS is not based on variance maximization but on the maximization/minimization of a generalized kurtosis, and it is invariant not only to orthogonal data transformations but also to any affine transformation.
The ```Julia``` ```ICSTools``` package brings the main functionalities of the ```R``` ```ICS``` package to ```Julia```, offering tools for identifying and selecting invariant coordinates in multivariate data. It includes various covariance estimators, transformation settings, and plotting utilities. Our extensive testing ensures results consistent with the ```R``` package, making it easy for users to transition from ```R``` to ```Julia``` or start fresh with ```ICS```.
The source code of this project is available at our GitHub Repository: [ICSTools](https://github.com/valentint/ICSTools.jl).
The complete documentation of the API with examples can be found at [```ICSTools``` Documentation](https://valentint.github.io/ICSTools.jl/dev/).
The results of the test coverage are available at
[```ICSTools``` Tests](https://app.codecov.io/github/valentint/ICSTools.jl?branch=main).


In [ ]:
import Pkg;
Pkg.add("Robustbase"); using Robustbase
Pkg.add("ICSTools"); using ICSTools
Pkg.add("Plots"); using Plots

   Resolving package versions...


## Getting started


In [ ]:
using ICSTools

# Load dataset
using Robustbase
X=wood[:,1:5];


In [ ]:
## Instantiate ICSModel object with all default parameters
ics = ICSModel();
show(ics)


In [ ]:
## Alternative instantiations:

## 1. With values for S1 and S2
ics1 = ICSModel(S1=cov2, S2=cov4);

## 2. With arguments for S2
ics2 = ICSModel(S1=cov2, S2=covW, S2_args=Dict{Symbol, Any}(:alpha=>1, :cf=>2));

## 3. With algorithm
ics3 = ICSModel(S1=cov2, S2=covW, algorithm="standard");


In [ ]:
## Fit the ICS model - equivalent of the function ICS-S3() from the R package ICS
ICSTools.fit!(ics, X);
show(ics)


In [ ]:
## Predict using the fitted model
scores=predict(ics, X);
scores


The following example illustrates how to plot the transformed/predicted data (invariant components) using ```component_plot2()``` and the kurtosis of the invariant components (corresponding to the eigenvalues of the joint diagonalization problem) using ```scree_plot()```.


In [ ]:
## scree plot and 2-dimensional component plot
using Plots
gr()
scree_plot(ics)
component_plot2(ics)                # by default the first two components
component_plot2(ics, select=[3,4])  # select components 3 and 4


## The scatters
ICS is based on simultaneously diagonalizing two different scatter matrices. The ICSTools package contains a number of implementations of scatter matrices, almost all scatters that are available in the different packages in ```R```.
Each scatter function takes a data matrix or a data frame and returns a structure with the following fields:
* (optional) location
* scatter matrix
* label


In [ ]:
## This is workaround for the errors in the show function for scatter
##in the current version of ICSTools

using ICSTools

function Base.show(io::IO, mime::MIME"text/plain", obj::ICSTools.Scatter)
    #   you can add IO options if you want
    #multiline = get(io, :multiline, true)
    #print_object(io, obj, multiline = multiline)

    println(io, "-> Scatter: " , obj.label)

    if !isnothing(obj.location)
        println(io, "Location:")
        println(IOContext(io, :compact=>true), obj.location)
    end

    println(io)
    println(io, "Scatter:")
    Base.show(io, mime, obj.scatter)
end


In [ ]:
using ICSTools

# Load dataset
using Robustbase
X=wood[:,1:5];

cov4(X)


Some scatter functions can take optional arguments. All functions are implemented in the ```ICSTools``` package, ```mcd_raw()``` and ```mcd_rew()``` use the Julia package ```Robustbase``` for the implementation of the FASTMCD algorithm.


In [ ]:
covW(X, alpha=2, cf=4)


When scatter functions are used as arguments to ```ICSModel()```, the optional arguments can be passed as data dictionaries:


In [ ]:
ics1 = ICSModel(S1=cov2, S2=covW, S2_args=Dict{Symbol, Any}(:alpha=>2, :cf=>4));

ics2 = ICSModel(S1=mcd_raw, S2=cov2, S1_args=Dict{Symbol, Any}(:nsamp=>1000, :alpha=>0.75));


Table 1. List of scatter functions implemented in the ```Julia```package ```ICSTool```.
| | Function | Location | Optional parameters |
| :--- | :---: | :---: |:--- |
| Covariance | ```cov2``` | Yes | - |
| Fourth-moment covariance | ```cov4``` | Yes | - |
| One-step M-estimator | ```covW``` | Yes | ```alpha```, ```cf``` |
| One-step Tyler shape matrix | ```covAxis``` | Yes | - |
| Multivariate t-distribution estimator | ```tM``` | Yes | ```alg```, ```mu_init```, ```V_init```, ```gamma_init```, ```eps```, ```maxiter``` |
| Cauchy location and scatter | ```mlc``` | Yes | ```alg```, ```mu_init```, ```V_init```, ```gamma_init```, ```eps```, ```maxiter``` |
| Raw MCD | ```mcd_raw``` | Yes | ```alpha```,```nsamp``` |
| Reweighted MCD | ```mcd_rwt``` | Yes | ```alpha```, ```nsamp``` |
| Pairwise one-step M-estimate | ```tcov``` | No | ```beta``` |
| Local shape scatter | ```lcov``` | No | ```proportion```, ```mscatter```, ```mcdalpha```, ```covstandard``` |



## Example applications
### The Penguins data set
Data on adult penguins covering three species found on three islands in the Palmer Archipelago, Antarctica, including their size (flipper length, body mass, bill dimensions), and sex. The data set ```penguins``` in ```R``` is a data frame with 344 rows and 8 variables. It is also available in ```Julia``` as a package ```PalmerPenguins.jl```.
Be careful there are missing values so we could consider only 342 observations. However, there are further 9 observations for which the variable ```sex``` has ```NA```, therefore it is better to work with the remaining 333 observations.
The idea of the penguins data set is to replace the well known ```iris``` data set which is used throughout data science, education statistical software and machine learning as test and illustration data set. The structure of ```penguins``` is similar to that of ```iris``` but still there are several essential differences: There are several grouping variables (species, island, sex and year), the sample sizes of the groups (152-68-124) are not equal as in iris (3x50), there are a few observations with missing values.
Table 1. Sample sizes for ```iris``` and ```penguins```. Data in penguins can be further grouped by island and study year.
| Iris species | Sample size | Penguin species | Female | Male | NA |
| :--- | :---: | :--- | :---: | :---: | :---: |
| setosa | 50 | Adélie | 73 | 73 | 6 |
| versicolor | 50 | Chinstrap | 34 | 34 | 0 |
| virginica | 50 | Gentoo | 58 | 61 | 5 |
We start by downloading the data set from ```R```:


In [ ]:
Pkg.add("DataFrames");
Pkg.add("RCall");            # To access R data sets; to perform tests

In [ ]:
## Load the penguins data from R
using DataFrames
using RCall
Robj = R"data('penguins', package='datasets'); x=penguins";

## Copy the contents of an R object into a corresponding canonical Julia type
penguins = rcopy(Robj);
size(penguins)
penguins = dropmissing(penguins); # drop the missing values
size(penguins)
X = penguins[:,3:6];              # select only the quantitative variables


In [ ]:
## Tabulate species by sex
gdf = groupby(penguins, [:species, :sex]);
tt = combine(gdf, nrow);
show(IOContext(stdout, :limit=>false), MIME"text/plain"(), tt)


Now we apply ICS with scaters ```tcov``` and ```cov2```, transform the model using the penguin data and plot the ```scree_plot``` (analogue to eigenvalues screeplot in PCA).


In [ ]:
using ICSTools
ics = ICSModel(S1=tcov, S2=cov2);
scores = fit_predict!(ics, X);


In [ ]:
using Plots
gr()
scree_plot(ics)


For convineance, to look at the data taking the grouping by species and by sex into account we convert the categorical arrays into string arrays.


In [ ]:
Pkg.add("CategoricalArrays");

In [ ]:
## Convert the categorical arrays species and sex to string arrays
using CategoricalArrays
species = unwrap.(penguins.species);
sex = unwrap.(penguins.sex);


Then we plot the first two IC components grouping the data by species (the different colours represent the different species). We see that the species are very well separated.


In [ ]:
## Convert the categorical arrays species and sex to string arrays
component_plot2(ics, clusters=species)


Next, we plot the fourth IC component grouping the data by sex . We see that the sex of the penguins is very well separated on the fourth IC.


In [ ]:
## Convert the categorical arrays species and sex to string arrays
component_plot2(ics, clusters=sex, select=[4])


Finally, we look at the pair plots of the IC scores of the penguins data.
This function (```component_plot()```) is not implemented in the ```ICSTools``` package, therefore we call directly the ```pairplot()``` function. The grouping is by species.

*Please note that the following code may not run reliably on Google Colab, as users can encounter difficulties when installing the GLMakie package in that environment.*

In [ ]:
Pkg.add("GLMakie"); using GLMakie       # for pairplots() and related
Pkg.add("PairPlots"); using PairPlots   # for pairplots() and related

In [ ]:
using DataFrames
using GLMakie
using PairPlots

ss = DataFrame(scores, ["IC$i" for i=1:size(scores,2)]);

fig=pairplot(ss[species .== "Adelie", :] =>
             (PairPlots.Scatter(markersize=10),
              PairPlots.MarginDensity()),
            ss[species .== "Gentoo", :] =>
             (PairPlots.Scatter(markersize=10),
              PairPlots.MarginDensity()),
            ss[species .== "Chinstrap", :] =>
             (PairPlots.Scatter(markersize=10),
              PairPlots.MarginDensity()),
          fullgrid=true)
save("images/penguins-plot-species.png", fig)


![](images/penguins-plot-species.png)
Now we repeat the same with grouping by sex.


In [ ]:
fig=pairplot(ss[sex .== "female", :] =>
             (PairPlots.Scatter(markersize=10),
              PairPlots.MarginDensity()),
            ss[sex .== "male", :] =>
             (PairPlots.Scatter(markersize=10),
              PairPlots.MarginDensity()),
          fullgrid=true)
save("images/penguins-plot-sex.png", fig)
